In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp, get_json_object, explode, first, explode_outer
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, DoubleType, TimestampType, IntegerType
import pyspark.sql.functions as F

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_encounter")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

In [ ]:
bronze_path = "../../data_lake/bronze/batch_data/resource_type=Encounter/"
silver_encounter_bundle_path = "../../data_lake/silver/silver_encounter_bundle/"

In [ ]:
general_coding_schema = StructType([
        StructField("system", StringType(), True),
        StructField("code", StringType(), True),
        StructField("display", StringType(), True)
    ])

type_schema = StructType([
        StructField("coding", ArrayType(general_coding_schema), True),
        StructField("text", StringType(), True)
    ])

time_schema = StructType([
        StructField("start", TimestampType(), True),
        StructField("end", TimestampType(), True)
    ])

reference_schema = StructType([
            StructField("reference", StringType(), True),
            StructField("display", StringType(), True)
        ])

participant_schema = StructType([
    StructField("type", ArrayType(type_schema), True),
    StructField("period", time_schema, True),
    StructField("individual", reference_schema, True)
])

In [ ]:
df_bronze = spark.read.format("parquet").load(bronze_path)

In [ ]:
df_inter = df_bronze.select(
    col("resource.id").alias("encounter_id"),
    col("resource.status").alias("status"),
    get_json_object(col("resource.class"), "$.code").alias("class_code"),
    explode_outer(from_json(col("resource.type"), ArrayType(type_schema))).alias("type"),
    get_json_object(col("resource.subject"), "$.reference").alias("patient_id"),
    explode_outer(from_json(col("resource.participant"), ArrayType(participant_schema))).alias("participants"),
    from_json(col("resource.period"), time_schema).alias("period"),
    explode_outer(from_json(col("resource.reasonCode"), ArrayType(type_schema))).alias("reason_code"),
    from_json(col("resource.hospitalization"), StructType([
        StructField("dischargeDisposition", type_schema, True)
    ])).alias("hospitalization"),
    explode_outer(from_json(col("resource.location"), ArrayType(StructType([
        StructField("location", reference_schema, True)
    ])))).alias("locations"),
        # col("resource.location"),
    from_json(col("resource.serviceProvider"), reference_schema).alias("service_provider"),
    col("input_file_name")
)

In [ ]:
df_inter2 = (
	df_inter.select(
		"encounter_id",
		"status",
		"class_code",
		col("type.coding")[0]["code"].alias("type_code"),
		col("type.coding")[0]["display"].alias("type_display"),
		"patient_id",
		col("participants.period.start").alias("participant_period_start"),
		col("participants.period.end").alias("participant_period_end"),
		F.split(col("participants.individual.reference"), "\\?")[0].alias("participant_individual_type"),
		F.split(col("participants.individual.reference"), "\\|")[1].alias("participant_individual_npi"),
		col("period.start").alias("period_start"),
		col("period.end").alias("period_end"),
		explode_outer(col("reason_code.coding")).alias("reason_coding"),
		explode_outer(col("hospitalization.dischargeDisposition.coding")).alias("discharge_coding"),
		explode_outer(col("participants.type.coding")).alias("participant_type"),
		F.split(col("locations.location.reference"), "\\|")[1].alias("location_id"),
		F.split(col("service_provider.reference"), "\\|")[1].alias("organization_org_id"),
		col("input_file_name"),
		current_timestamp().alias("silver_timestamp")
	).select(
		"encounter_id",
		"status",
		"class_code",
		"type_code",
		"type_display",
		"patient_id",
		"participant_period_start",
		"participant_period_end",
		"participant_individual_type",
		"participant_individual_npi",
		"period_start",
		"period_end",
		col("reason_coding.code").alias("reason_code"),
		col("reason_coding.display").alias("reason_display"),
		"location_id",
		"organization_org_id",
		"input_file_name",
		"silver_timestamp",
		col("discharge_coding.code").alias("hospitalization_discharge_disposition_code"),
		col("discharge_coding.display").alias("hospitalization_discharge_disposition_display"),
		col("participant_type.code").alias("participant_type_code"),
		col("participant_type.display").alias("participant_type_display")
	).withColumn("participant_type_code", explode_outer(col("participant_type_code")))
    .withColumn("participant_type_display", explode_outer(col("participant_type_display")))
)


In [ ]:
df_inter2.write.mode("overwrite").format("parquet").save(silver_encounter_bundle_path)